# Fase de modelado

## Dataset para el modelo R

- **X_train_final**: contiene los datos de entrada del conjunto de entrenamiento (imputados, codificados y normalizados)

- **X_test_final**: contiene los datos de entrada del conjunto de test (imputados, codificados y normalizados)

- **y_train**: contiene la variable target ('Riesgo') para el conjunto de entrenamiento. Variable dicotómica (si/no)

- **y_test**: contiene la variable target ('Riesgo') para el conjunto de test

In [1]:
import sys
sys.path.append('..')

In [2]:
# Import basic
import pandas as pd
import numpy as np

In [3]:
# Import PyWin 
from pywinEA.algorithm import GA, NSGA2, SPEA2
from pywinEA.population import Population, BlockPopulation
from pywinEA.fitness import MonoObjectiveCV
from pywinEA.operators import TournamentSelection, RouletteWheel
from pywinEA.wrapper import Parallel, IslandModel
from pywinEA.visualization import Plotter, GAevaluator, MOAevaluator
from pywinEA.dataset import load_demo

In [4]:
# Import Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from collections import Counter
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)

In [5]:
# Data loading 
# R main model dataset (basales + diagnosis)

X_train = pd.read_csv('X_train_final_dist.csv')
X_test = pd.read_csv('X_test_final_dist.csv')
y_train = pd.read_csv('y_train.csv').squeeze('columns')  # Devuelve Series
y_test = pd.read_csv('y_test.csv').squeeze('columns')


In [6]:
X_train.columns

Index(['etnia_caucasica', 'etnia_asiatica', 'etnia_norteAfri', 'etnia_hispana',
       'etnia_otra', 'concep_espont', 'concep_insem', 'concep_fiv',
       'edad_materna', 'Peso', 'Talla', 'IMC', 'paridad', 'p_vaginal',
       'cesareas', 'abortos', 'riesgo_elevado', 'riesgo_moderado', 'EG_PE',
       'sflt1', 'PlGF', 'ratio', 'sFlt1MoM', 'PlGFMoM', 'ratioMoM', 'TAS_Incl',
       'TAD_Incl', 'TAM_Incl', 'EG_eco_incl', 'PFE_eco_incl', 'PFEp_eco_incl',
       'IPAUti_eco_incl', 'IPAUtd_eco_incl', 'IPmAUt_eco_incl',
       'IPmAUtMoM_eco_incl', 'IPmAUtp_eco_incl', 'IPACM_eco_incl',
       'IPACMp_eco_incl', 'IPAU_eco_incl', 'IPAUp_eco_incl', 'NumFar',
       'PE_previa', 'AAS', 'Heparina', 'RatioEscala', 'PlGFescala',
       'CIRestadio', 'ovodon', 'ht_cronica', 'nefropatia', 'dm_pregest',
       'trombofilia', 'LES', 'Em_mayor40', 'Nulipara_o_10omasañosparto',
       'IMCmayor35', 'af_pe', 'Fumadora', 'sFLt195', 'PLGF5', 'IPACMp5_eco1',
       'IPAUp95_eco1', 'sexo_rn', 'fenotipo_label'],

In [8]:
# filtro previo de variables
# quito las que tienen mucha variabilidad: sflt-1, PlGF, ratio y PFE
selected = ['etnia_caucasica', 'etnia_asiatica', 'etnia_norteAfri', 'etnia_hispana',
       'etnia_otra', 'concep_espont', 'concep_insem', 'concep_fiv',
       'edad_materna', 'Peso', 'Talla', 'IMC', 'paridad', 'p_vaginal',
       'cesareas', 'abortos', 'riesgo_elevado', 'riesgo_moderado', 'EG_PE',
       'sflt1', 'PlGF', 'ratio', 'sFlt1MoM', 'PlGFMoM', 'ratioMoM', 'TAS_Incl',
       'TAD_Incl', 'TAM_Incl', 'EG_eco_incl', 'PFE_eco_incl', 'PFEp_eco_incl',
       'IPAUti_eco_incl', 'IPAUtd_eco_incl', 'IPmAUt_eco_incl',
       'IPmAUtMoM_eco_incl', 'IPmAUtp_eco_incl', 'IPACM_eco_incl',
       'IPACMp_eco_incl', 'IPAU_eco_incl', 'IPAUp_eco_incl', 'NumFar',
       'ovodon', 'PE_previa', 'ht_cronica', 'nefropatia', 'dm_pregest',
       'trombofilia', 'LES', 'Em_mayor40', 'Nulipara_o_10omasañosparto',
       'IMCmayor35', 'af_pe', 'AAS', 'Heparina', 'Fumadora', 'RatioEscala',
       'sFLt195', 'PlGFescala', 'PLGF5', 'IPACMp5_eco1', 'IPAUp95_eco1',
       'CIRestadio', 'sexo_rn']

X_train = X_train[selected]
X_test = X_test[selected]

In [8]:
# Converting categorical features (both binary and polytomic) to "category" type
cat_binarias = ['etnia_caucasica', 'etnia_asiatica', 'etnia_norteAfri', 'etnia_hispana',
       'etnia_otra', 'concep_espont', 'concep_insem', 'concep_fiv','Em_mayor40',
       'IMCmayor35', 'sFLt195', 'PLGF5',
       'IPACMp5_eco1', 'IPAUp95_eco1', 'ovodon', 'PE_previa', 'ht_cronica',
       'nefropatia', 'dm_pregest', 'trombofilia', 'LES',
       'Nulipara_o_10omasañosparto', 'af_pe', 'AAS', 'Heparina', 'Fumadora', 'sexo_rn']
cat_polit = ['RatioEscala', 'PlGFescala','PE_previa',  'AAS', 'Heparina', 'CIRestadio']

for col in cat_binarias:
    X_train[col] = X_train[col].astype('category')
    
for col in cat_polit:
    X_train[col] = X_train[col].astype('category')

In [9]:
# Converting categorical features (both binary and polytomic) to "category" type
cat_binarias = ['etnia_caucasica', 'etnia_asiatica', 'etnia_norteAfri', 'etnia_hispana',
       'etnia_otra', 'concep_espont', 'concep_insem', 'concep_fiv','Em_mayor40',
       'IMCmayor35', 'sFLt195', 'PLGF5',
       'IPACMp5_eco1', 'IPAUp95_eco1', 'ovodon', 'PE_previa', 'ht_cronica',
       'nefropatia', 'dm_pregest', 'trombofilia', 'LES',
       'Nulipara_o_10omasañosparto', 'af_pe', 'AAS', 'Heparina', 'Fumadora', 'sexo_rn']
cat_polit = ['RatioEscala', 'PlGFescala','PE_previa',  'AAS', 'Heparina', 'CIRestadio']

for col in cat_binarias:
    X_test[col] = X_test[col].astype('category')
    
for col in cat_polit:
    X_test[col] = X_test[col].astype('category')

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
X_train.dtypes

## Variables basales

Las selecciono de todo el conjunto de datos, para que así sean comparables (esto puedo hacerlo porque en las basales como tal no hay missing values, así que no tengo por qué repetir el proceso de imputación solo con esas variables). Además para la normalización y la codificación se tiene en cuenta solo la propia variable y no el resto, y como el nº de pacientes es el mismo, nada cambia.

### Conjunto de train

- **Shape basales**: 277x30

- **Shape basales + diagnóstico**: 277 x 59

### Conjunto de test

- **Shape basales**: 70x30

- **Shape basales + diagnóstico**: 70 x 59

In [9]:
X_TrainData = X_train.values
y_TrainData = y_train.values
features = X_train.columns.tolist()
print(X_TrainData)
print(y_TrainData)
print(features)

[[ 0.          0.          0.         ...  0.         -0.97179797
   0.        ]
 [ 0.          0.          0.         ...  1.          1.28610887
   0.        ]
 [ 0.          0.          1.         ...  0.          0.15715545
   1.        ]
 ...
 [ 0.          0.          0.         ...  0.         -0.97179797
   1.        ]
 [ 0.          0.          0.         ...  0.          0.15715545
   1.        ]
 [ 0.          0.          0.         ...  0.          0.15715545
   1.        ]]
[0 1 1 1 0 0 0 1 0 1 1 0 1 1 0 1 1 0 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0
 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 1 1 1 1 1 1 0 0 0 1 0 0 1 1 0 0 0 0 1
 1 0 1 0 0 0 0 1 0 1 0 0 1 1 0 1 0 0 1 1 1 0 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0
 1 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 1 0 1 1 0 0 1 0 0 0 0 0 0
 1 1 1 1 1 0 0 0 1 0 0 0 1 1 0 1 1 0 1 1 0 0 1 0 0 1 1 0 0 0 0 0 0 0 0 1 0
 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 1 0 1 0 1 0 1 1
 0 1 0 0 0 0 0 0 0 1 1 0 0 0 1 0 1 0 1 0 0 0 1 1 0 0 0 1 1

In [10]:
# GA fixed parameters
# Estos parámetros se elegin manualmente en función de tus datos (yo suelo probar varias combinaciones de estos parámetros)
POPULATION_SIZE = 120
GENERATIONS = 120
MUTATION_RATE = 0.03
ELITISM = 0.3
ANNIHILATION = 0.3
FILL_WITH_ELITE = 0.5
MEASURE = 'f1'

tournament = TournamentSelection(k=2, winners=1, replacement=False)

# Scikit-learn cross-validation
rep_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=1234) # yo uso este tipo de validación porque me parece más robusta

# Contador de folds
cnt = 1

# Aplicar validación cruzada sobre el conjunto de entrenamiento
for train_index, val_index in rep_cv.split(X_train, y_train):
    print(f'Fold: {cnt}, Train set: {len(train_index)}, Validation set: {len(val_index)}')
    cnt += 1

Fold: 1, Train set: 281, Validation set: 71
Fold: 2, Train set: 281, Validation set: 71
Fold: 3, Train set: 282, Validation set: 70
Fold: 4, Train set: 282, Validation set: 70
Fold: 5, Train set: 282, Validation set: 70
Fold: 6, Train set: 281, Validation set: 71
Fold: 7, Train set: 281, Validation set: 71
Fold: 8, Train set: 282, Validation set: 70
Fold: 9, Train set: 282, Validation set: 70
Fold: 10, Train set: 282, Validation set: 70
Fold: 11, Train set: 281, Validation set: 71
Fold: 12, Train set: 281, Validation set: 71
Fold: 13, Train set: 282, Validation set: 70
Fold: 14, Train set: 282, Validation set: 70
Fold: 15, Train set: 282, Validation set: 70
Fold: 16, Train set: 281, Validation set: 71
Fold: 17, Train set: 281, Validation set: 71
Fold: 18, Train set: 282, Validation set: 70
Fold: 19, Train set: 282, Validation set: 70
Fold: 20, Train set: 282, Validation set: 70
Fold: 21, Train set: 281, Validation set: 71
Fold: 22, Train set: 281, Validation set: 71
Fold: 23, Train set

In [11]:
# Scikit-learn estimator


#SVM = SVC(kernel= 'poly', C = 0.12, degree = 2, gamma = 0.04, class_weight='balanced', probability=True, random_state=42)
SVM = SVC(probability=False, class_weight='balanced', random_state=42)

KNN = KNeighborsClassifier()

LR = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
#LR1 = LogisticRegression(penalty = 'l2', solver = 'liblinear', C = 0.007, class_weight='balanced', max_iter=1000, random_state=42)

#LR2 = LogisticRegression(penalty = 'elasticnet', solver = 'saga', C = 0.11, l1_ratio = 0.57, class_weight='balanced', max_iter=1000, random_state=42)

#TD = DecisionTreeClassifier(max_depth = 2, min_samples_split = 14, min_samples_leaf = 4, criterion = 'entropy', class_weight='balanced', random_state=42)

TD = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)

ETC = ExtraTreesClassifier(n_estimators=200, max_depth=3, class_weight="balanced",random_state=42)

RFC = RandomForestClassifier(n_estimators=200,max_depth=3, class_weight="balanced",random_state=42)

GB = GradientBoostingClassifier(max_depth = 3, random_state=42)

Gaus = GaussianNB()

In [12]:
# Create fitness function

fitness_SVM = MonoObjectiveCV(
    estimator=SVM,
    cv=rep_cv,
    score=MEASURE
)
fitness_KNN = MonoObjectiveCV(
    estimator=KNN,
    cv=rep_cv,
    score=MEASURE
)
fitness_LR = MonoObjectiveCV(
    estimator=LR,
    cv=rep_cv,
    score=MEASURE
)
fitness_TD = MonoObjectiveCV(
    estimator=TD,
    cv=rep_cv,
    score=MEASURE
)
fitness_ETC = MonoObjectiveCV(
    estimator=ETC,
    cv=rep_cv,
    score=MEASURE
)
fitness_RFC = MonoObjectiveCV(
    estimator=RFC,
    cv=rep_cv,
    score=MEASURE
)
fitness_GB = MonoObjectiveCV(
    estimator=GB,
    cv=rep_cv,
    score=MEASURE
)
fitness_Gaus = MonoObjectiveCV(
    estimator=Gaus,
    cv=rep_cv,
    score=MEASURE
)

In [14]:
# Option 1: using the previously defined AG parameters
basic1 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_SVM, 
    annihilation=ANNIHILATION, fill_with_elite=FILL_WITH_ELITE, elitism=ELITISM, mutation_rate=MUTATION_RATE, 
    selection=tournament, random_state=1234, id="reentRBasDiag_SVM")
basic2 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_KNN, 
    annihilation=ANNIHILATION, fill_with_elite=FILL_WITH_ELITE, elitism=ELITISM, mutation_rate=MUTATION_RATE, 
    selection=tournament, random_state=1234, id="reentRBasDiag_KNN")
basic3 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_LR, 
    annihilation=ANNIHILATION, fill_with_elite=FILL_WITH_ELITE, elitism=ELITISM, mutation_rate=MUTATION_RATE, 
    selection=tournament, random_state=1234, id="reentRBasDiag_LR")
basic4 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_TD, 
    annihilation=ANNIHILATION, fill_with_elite=FILL_WITH_ELITE, elitism=ELITISM, mutation_rate=MUTATION_RATE, 
    selection=tournament, random_state=1234, id="reentRBasDiag_TD")
basic5 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_ETC, 
    annihilation=ANNIHILATION, fill_with_elite=FILL_WITH_ELITE, elitism=ELITISM, mutation_rate=MUTATION_RATE, 
    selection=tournament, random_state=1234, id="reentRBasDiag_ETC")
basic6 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_RFC, 
    annihilation=ANNIHILATION, fill_with_elite=FILL_WITH_ELITE, elitism=ELITISM, mutation_rate=MUTATION_RATE, 
    selection=tournament, random_state=1234, id="reentRBasDiag_RFC")
basic7 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_GB, 
    annihilation=ANNIHILATION, fill_with_elite=FILL_WITH_ELITE, elitism=ELITISM, mutation_rate=MUTATION_RATE, 
    selection=tournament, random_state=1234, id="reentRBasDiag_GB")
basic8 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_Gaus, 
    annihilation=ANNIHILATION, fill_with_elite=FILL_WITH_ELITE, elitism=ELITISM, mutation_rate=MUTATION_RATE, 
    selection=tournament, random_state=1234, id="reentRBasDiag_Gaus")

In [11]:
# Option 2: Testing different values for the annihilation, fill with elite, elitism and mutation rate parameters
basic1 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_SVM, 
    annihilation=0.1, fill_with_elite=0.5, elitism=0.2, mutation_rate=0.05, 
    selection=tournament, random_state=1234, id="RBasDiagsSC_monoSVM1")
basic2 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_SVM, 
    annihilation=0.1, fill_with_elite=0.5, elitism=0.2, mutation_rate=0.02, 
    selection=tournament, random_state=1234, id="RBasDiagsSC_monoSVM2")
basic3 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_SVM, 
    annihilation=0.1, fill_with_elite=0.5, elitism=0.2, mutation_rate=0.1, 
    selection=tournament, random_state=1234, id="RBasDiagsSC_monoSVM3")
basic4 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_SVM, 
    annihilation=0.3, fill_with_elite=0.3, elitism=0.3, mutation_rate=0.03, 
    selection=tournament, random_state=1234, id="RBasDiagsSC_monoSVM4")
basic5 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_SVM, 
    annihilation=0.3, fill_with_elite=0.5, elitism=0.3, mutation_rate=0.02, 
    selection=tournament, random_state=1234, id="RBasDiagsSC_monoSVM5")
basic6 = GA(
    population=Population(size=POPULATION_SIZE), generations=GENERATIONS, fitness=fitness_SVM, 
    annihilation=0.3, fill_with_elite=0.5, elitism=0.3, mutation_rate=0.1, 
    selection=tournament, random_state=1234, id="RBasDiagsSC_monoSVM6")

In [12]:
# Set feature names 
basic1.set_features(features)
basic2.set_features(features)
basic3.set_features(features)
basic4.set_features(features)
basic5.set_features(features)
basic6.set_features(features)
basic7.set_features(features)
basic8.set_features(features)

In [ ]:
try:
	basic1 = basic1.fit(X_TrainData, y_TrainData)
	basic1.save("reentRBasDiag_SVM")
except:
	print("Error inesperado:", sys.exc_info())
	error=open("errores.txt","a")
	error.write(f'Error producido en: MR(0.01)\n')
	error.write(f'Nombre del error: {sys.exc_info()}\n\n\n\n')
	error.close()

In [ ]:
try:
	basic2 = basic2.fit(X_TrainData, y_TrainData)
	basic2.save("reentRBasDiag_KNN")
except:
	print("Error inesperado:", sys.exc_info())
	error=open("errores.txt","a")
	error.write(f'Error producido en: MR(0.01)\n')
	error.write(f'Nombre del error: {sys.exc_info()}\n\n\n\n')
	error.close()

In [ ]:
try:
	basic3 = basic3.fit(X_TrainData, y_TrainData)
	basic3.save("reentRBasDiag_LR")
except:
	print("Error inesperado:", sys.exc_info())
	error=open("errores.txt","a")
	error.write(f'Error producido en: MR(0.01)\n')
	error.write(f'Nombre del error: {sys.exc_info()}\n\n\n\n')
	error.close()

In [ ]:
try:
	basic4 = basic4.fit(X_TrainData, y_TrainData)
	basic4.save("reentRBasDiag_TD")
except:
	print("Error inesperado:", sys.exc_info())
	error=open("errores.txt","a")
	error.write(f'Error producido en: MR(0.01)\n')
	error.write(f'Nombre del error: {sys.exc_info()}\n\n\n\n')
	error.close()

In [ ]:
try:
	basic5 = basic5.fit(X_TrainData, y_TrainData)
	basic5.save("reentRBasDiag_ETC")
except:
	print("Error inesperado:", sys.exc_info())
	error=open("errores.txt","a")
	error.write(f'Error producido en: MR(0.01)\n')
	error.write(f'Nombre del error: {sys.exc_info()}\n\n\n\n')
	error.close()

In [ ]:
try:
	basic6 = basic6.fit(X_TrainData, y_TrainData)
	basic6.save("reentRBasDiag_RFC")
except:
	print("Error inesperado:", sys.exc_info())
	error=open("errores.txt","a")
	error.write(f'Error producido en: MR(0.01)\n')
	error.write(f'Nombre del error: {sys.exc_info()}\n\n\n\n')
	error.close()

In [ ]:
try:
	basic7 = basic7.fit(X_TrainData, y_TrainData)
	basic7.save("reentRBasDiag_GB")
except:
	print("Error inesperado:", sys.exc_info())
	error=open("errores.txt","a")
	error.write(f'Error producido en: MR(0.01)\n')
	error.write(f'Nombre del error: {sys.exc_info()}\n\n\n\n')
	error.close()

In [ ]:
try:
	basic8 = basic8.fit(X_TrainData, y_TrainData)
	basic8.save("reentRBasDiag_Gaus")
except:
	print("Error inesperado:", sys.exc_info())
	error=open("errores.txt","a")
	error.write(f'Error producido en: MR(0.01)\n')
	error.write(f'Nombre del error: {sys.exc_info()}\n\n\n\n')
	error.close()

# Result's analysis

In [13]:
# Para los modelos descargados del servidor
from pathlib import Path

MODELS_DIR = Path("/Desktop/preeclampsia_codes/jupyter/modelos_validacion/reentrenamiento/PyWinModels")

model_path = MODELS_DIR / "reentRBasDiag_SVM"
ga_basic = GA.load(str(model_path))   # o basic1.load(...) según tu API concreta

FileNotFoundError: [Errno 2] No such file or directory: './_PyWinModels//Desktop/preeclampsia_codes/jupyter/modelos_validacion/reentrenamiento/PyWinModels/reentRBasDiag_SVM'

In [17]:
# Loading the model that we want to analyze
ga_basic = basic1.load("reentRBasDiag_SV")
#ga_basic = basic2.load("/desktop/preeclampsia_codes/jupyter/_PyWinModels/RBasDiagsSC_monoKNN")

In [ ]:
# All attributes
print("ga_basic.get_current_generation OUT:", ga_basic.get_current_generation, end="\n\n")
print("ga_basic.best_features OUT:", ga_basic.best_features, end="\n\n")
print("ga_basic.population OUT:", ga_basic.population, end="\n\n")
print("ga_basic.population_fitness OUT:", ga_basic.population_fitness, end="\n\n")
print("ga_basic.best_performance OUT:", ga_basic.best_performance, end="\n\n")
print("ga_basic.fitness OUT:", ga_basic.fitness, end="\n\n")
print("ga_basic.generations OUT:", ga_basic.generations, end="\n\n")
print("ga_basic.selection OUT:", ga_basic.selection, end="\n\n")
print("ga_basic.elitism OUT:", ga_basic.elitism, end="\n\n")
print("ga_basic.elitism_rate OUT:", ga_basic.elitism_rate, end="\n\n")
print("ga_basic.annihilation OUT:", ga_basic.annihilation, end="\n\n")
print("ga_basic.annihilation_rate OUT:", ga_basic.annihilation_rate, end="\n\n")
print("ga_basic.fill_with_elite OUT:", ga_basic.fill_with_elite, end="\n\n")
print("ga_basic.mutation OUT:", ga_basic.mutation, end="\n\n")
print("ga_basic.mutation_rate OUT:", ga_basic.mutation_rate, end="\n\n")
print("ga_basic.imputer OUT:", ga_basic.imputer, end="\n\n")
print("ga_basic.crossover OUT:", ga_basic.crossover, end="\n\n")
print("ga_basic.positive_class OUT:", ga_basic.positive_class, end="\n\n")
print("ga_basic.random_state OUT:", ga_basic.random_state, end="\n\n")
print("ga_basic.id OUT:", ga_basic.id, end="\n\n")

In [ ]:
# To access to all the solutions
print("ga_basic.population.individuals OUT:", ga_basic.population.individuals)

In [ ]:
individuals = ga_basic.population.individuals

# Filtering the individuals (combinations of features) with a fitness > 0.7 (considering that the best individual (final selected AG values) has a fitnees of 0.7128557394297599 
filtered_individuals = [ind for ind in individuals if ind.fitness > 0.63]

# Printing the filtered individuals
for ind in filtered_individuals:
    print(f"Filtered individual - Features: {ind.features}, Fitness: {ind.fitness}")

In [ ]:
# 'features' es la lista de nombres de columnas del dataset
# y que cada ind.features contiene índices (por ejemplo [3, 7, 12, 19])

features = X_train.columns.tolist()

for ind in filtered_individuals:
    # Convertir índices a nombres de variables
    selected_feature_names = [features[i] for i in ind.features]

    # Contar el número de variables seleccionadas
    num_features = len(selected_feature_names)

    print(f"Filtered individual - Fitness: {ind.fitness:.3f}")
    print(f"Nº de features seleccionadas: {num_features}")
    print(f"Features: {selected_feature_names}")
    print("-" * 80)


In [ ]:
variable_names = X_train.columns


# Encontrar el individuo con el fitness máximo
max_fitness_individual = max(individuals, key=lambda ind: ind.fitness)

# Encontrar el individuo con el fitness mínimo
min_fitness_individual = min(individuals, key=lambda ind: ind.fitness)

# Mapear los índices de features a nombres
max_features_names = [variable_names[i] for i in max_fitness_individual.features]
min_features_names = [variable_names[i] for i in min_fitness_individual.features]

# Obtener los valores de fitness
max_fitness = max_fitness_individual.fitness
min_fitness = min_fitness_individual.fitness

# Imprimir los resultados
print(f"Highest fitness: {max_fitness}, Features: {max_features_names}")
print(f"Lowest fitness: {min_fitness}, Features: {min_features_names}")

In [ ]:
# Lista de individuos ordenados de mayor a menor fitness, con los nombres de sus variables. 
# Solo se imprimen los que NO contienen las variables de las uterinas
# Lista de nombres de variables (en el orden de los índices)
variable_names = df.columns

# Variables a excluir
excluded_variables = {'IPAUti_eco_incl', 'IPAUtd_eco_incl',
       'IPmAUt_eco_incl', 'IPmAUtMoM_eco1', 'IPmAUtp_eco1'}

# Filtrando y ordenando los individuos que cumplen el criterio
filtered_individuals = sorted(
    [
        ind for ind in individuals
        if not excluded_variables.intersection({variable_names[i] for i in ind.features})
    ],
    key=lambda ind: ind.fitness,
    reverse=True  # Orden descendente
)

# Imprimiendo los individuos filtrados con nombres de variables
if filtered_individuals:
    for ind in filtered_individuals:
        # Mapear índices de features a nombres de variables
        feature_names = [variable_names[i] for i in ind.features]
        print(f"Fitness: {ind.fitness}, Features: {feature_names}")
else:
    print('No individuals met the criteria.')

In [ ]:
Plotter.plot_evolution(ga_basic)

In [24]:
# Create an evaluator instance to represent results
evaluator = GAevaluator(ga_basic)

In [ ]:
# Evaluate results
evaluator.metrics(cv=5, reps=5, random_state=1234)

In [ ]:
evaluator.training_confusion_matrix()

In [ ]:
# Explore confusion matrix on training data using 10 repeats of 5-CV
evaluator.test_confusion_matrix(cv=5, reps=5, random_state=1234)

In [ ]:
# ROC using 5 repeats of 5-CV
evaluator.roc(cv=5, reps=5)

In [ ]:
# Plot learning curve
evaluator.plot_learning_curve(cv=5)

## Test set evaluation

In [ ]:
# Access to the best model 
# IMPORTANT: If we analyze the test set with scikit-learn the results are slightly different. It is better to access to the best model directly with the GA 
ga_basic._best_model

In [ ]:
sel = ['Nulipara_o_10omasañosparto', 'etnia_norteAfri', 'abortos', 'sflt1', 'ht_cronica', 'TAM_Incl', 'PlGFescala', 'trombofilia', 'EG_PE', 'Talla', 'TAD_Incl', 'CIRestadio', 'EG_eco_incl', 'etnia_otra', 'NumFar', 'af_pe']
X_test_sel = X_test[sel]
X_test_sel

In [ ]:
X_test_sel = X_test_sel.values
X_test_sel

In [ ]:
X_train_sel = X_train[sel]
X_train_sel

In [ ]:
X_train_sel = X_train_sel.values
X_train_sel

In [ ]:
model= LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train_sel, y_TrainData)
y_pred=model.predict(X_test_sel)
print(y_pred)
print(len(y_pred))

In [ ]:
model=ga_basic._best_model
y_pred=model.predict(X_test_sel)
print(y_pred)
print(len(y_pred))

In [85]:
from sklearn.metrics import confusion_matrix

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn)

In [86]:
# Calculate accuracy, precision, recall, f1-score, and kappa score
from sklearn import metrics
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import auc
from sklearn.metrics import make_scorer, confusion_matrix

acc = metrics.accuracy_score(y_test, y_pred)
prec = metrics.precision_score(y_test, y_pred)
rec = metrics.recall_score(y_test, y_pred)
f1 = metrics.f1_score(y_test, y_pred)

# Computing specificity, negative predictive value and confusion matrix
specificity = specificity_score(y_test, y_pred)
npv = npv_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)


# Generating a classifier without training
ns_probs = [0 for _ in range(len(y_test))]
# Predecimos las probabilidades
lr_probs = model.predict_proba(X_test_sel)
# 1 class probabilies
lr_probs = lr_probs[:, 1]
# AUC computation
ns_auc = roc_auc_score(y_test, ns_probs)
lr_auc = roc_auc_score(y_test, lr_probs)
# ROC curves
ns_fpr, ns_tpr, _ = roc_curve(y_test, ns_probs)
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_probs)
# Precision and recall curve
lr_precision, lr_recall, _ = precision_recall_curve(y_test, lr_probs)
lr_f1, lr_auc_pr = metrics.f1_score(y_test, y_pred), auc(lr_recall, lr_precision)
no_skill = len(y_test[y_test==1]) / len(y_test)

In [ ]:
# Printing the results for the test set
print(f"Accuracy: {acc:.3f}")
print(f"Precision (PPV): {prec:.3f}")
print(f"Recall: {rec:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"NPV: {npv:.3f}")
print(cm)

In [ ]:
# ROC curve and Precision-Recall curve for the test set

# ROC curve
plt.plot(ns_fpr, ns_tpr, linestyle='--', label='AUC = %.2f' % (ns_auc))
plt.plot(lr_fpr, lr_tpr, marker='.', label='AUC = %.3f' % (lr_auc))
# Axis labels
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.legend()
plt.show()
print('Sin entrenar: ROC AUC=', ns_auc) 
print('Entrenado con clasificador: ROC AUC=', lr_auc)

# Precision-Recall curve
plt.plot([0, 1], [no_skill,no_skill], linestyle='--', label='Sin entrenar')
plt.plot(lr_recall, lr_precision, marker='.', label='AUC = %.3f' % (lr_auc_pr))
# Axis labels
plt.xlabel('Sensibilidad')
plt.ylabel('Precisión')
plt.legend()
print('Modelo entrenado: f1= auc=', (lr_f1, lr_auc_pr))